# Exp 3-v9 🎯 — TF-IDF + **스케일 정규화 핸드크래프트 피처** + Sweep

## v8 문제 분석
- Dev 70.05% → Test 69.13% (gap 0.92%p)
- **원인**: 핸드크래프트 피처 스케일 불균형
  - TF-IDF: 0.0 ~ 1.0
  - 단어 수: 0 ~ 50  ← **50배 큼** → MLP 과도한 의존 → 일반화 저하

## v9 핵심 개선
- `StandardScaler`로 핸드크래프트 피처 정규화 → 평균 0, 표준편차 1
- TF-IDF와 동일한 스케일에서 학습 → dev-test gap 감소

## 예상: Dev-Test gap 감소 → Test **70%+** 🎯

In [ ]:
!pip install datasets wandb scikit-learn -q

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from scipy.sparse import hstack, csr_matrix
import numpy as np, copy, re, wandb
SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

Device:cuda


In [ ]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f'Train:{len(train_data)}|Dev:{len(dev_data)}|Test:{len(test_data)}|Classes:{output_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_df.csv: 0.00B [00:00, ?B/s]

val_df.csv: 0.00B [00:00, ?B/s]

test_df.csv: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/5206 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31232 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5206 [00:00<?, ? examples/s]

Train:31232|Dev:5205|Test:5205|Classes:3


In [ ]:
def preprocess_text(text):
    text=text.lower()
    text=re.sub(r'http\S+|www\S+','',text)
    text=text.replace('`',"'")
    text=text.replace('****',' bad ').replace('***',' bad ')
    text=re.sub(r'!{3,}',' verymuch ! ',text)
    text=re.sub(r'(.)\1{3,}',r'\1\1',text)
    text=re.sub(r"won't",'will not',text)
    text=re.sub(r"can't",'cannot',text)
    text=re.sub(r"n't",' not',text)
    text=re.sub(r"'re",' are',text)
    text=re.sub(r"'ve",' have',text)
    text=re.sub(r"'ll",' will',text)
    text=re.sub(r"'d",' would',text)
    text=re.sub(r"'m",' am',text)
    for pat,rep in [
        (r'\bidk\b','i do not know'),(r'\bur\b','your'),
        (r'\bnaw\b','no'),(r'\bgonna\b','going to'),
        (r'\bwanna\b','want to'),(r'\blol\b','laughing'),
        (r'\bomg\b','oh my god'),(r'\bwtf\b','what the'),
        (r'\bugh\b','disgusting'),(r'\btho\b','though'),
        (r'\bkinda\b','kind of'),(r'\bcuz\b','because'),
        (r'\bsoo+\b','so'),(r'\bthx\b','thanks'),
        (r'\byep\b','yes'),(r'\byup\b','yes'),
        (r'\bnope\b','no'),(r'\btbh\b','to be honest'),
        (r'\bimo\b','in my opinion'),
    ]:
        text=re.sub(pat,rep,text)
    return text

def extract_handcraft(texts):
    features=[]
    for text in texts:
        t=str(text); tl=t.lower(); words=t.split()
        features.append([
            t.count('!'),
            t.count('?'),
            sum(1 for w in words if w.isupper() and len(w)>1),
            len(words),
            int(bool(re.search(r'http\S+',tl))),
            int(any(e in tl for e in [':)',':(',':d',':/','haha','hehe','lmao'])),
        ])
    return np.array(features,dtype=np.float32)

# TF-IDF
vectorizer=TfidfVectorizer(max_features=30000,preprocessor=preprocess_text,min_df=2)
vectorizer.fit(train_data['text'])

# 핸드크래프트 피처 추출 후 StandardScaler 적용 (train 기준으로 fit)
hc_train_raw=extract_handcraft(train_data['text'])
hc_dev_raw=extract_handcraft(dev_data['text'])
hc_test_raw=extract_handcraft(test_data['text'])

scaler=StandardScaler()
hc_train_scaled=scaler.fit_transform(hc_train_raw)   # train으로만 fit!
hc_dev_scaled=scaler.transform(hc_dev_raw)
hc_test_scaled=scaler.transform(hc_test_raw)

# 스케일 확인
feat_names=['느낌표','물음표','ALL_CAPS','단어수','URL','이모티콘']
print('핸드크래프트 피처 정규화 전/후 범위:')
for i,name in enumerate(feat_names):
    before=f'{hc_train_raw[:,i].min():.1f}~{hc_train_raw[:,i].max():.1f}'
    after=f'{hc_train_scaled[:,i].min():.2f}~{hc_train_scaled[:,i].max():.2f}'
    print(f'  {name}: {before} → {after}')

def build_features(tfidf_mat, hc_scaled):
    combined=hstack([tfidf_mat, csr_matrix(hc_scaled)])
    return torch.FloatTensor(combined.toarray()).to(device)

train_t=build_features(vectorizer.transform(train_data['text']),hc_train_scaled)
dev_t=build_features(vectorizer.transform(dev_data['text']),hc_dev_scaled)
test_t=build_features(vectorizer.transform(test_data['text']),hc_test_scaled)
dev_labels_t=torch.tensor(dev_data['label'],dtype=torch.long).to(device)
input_size=train_t.shape[1]
print(f'\n입력 크기: {input_size} (TF-IDF 30000 + 핸드크래프트 6, 정규화됨)')

핸드크래프트 피처 정규화 전/후 범위:
  느낌표: 0.0~59.0 → -0.40~47.66
  물음표: 0.0~98.0 → -0.17~115.83
  ALL_CAPS: 0.0~60.0 → -0.23~51.73
  단어수: 1.0~391.0 → -0.95~20.90
  URL: 0.0~1.0 → -0.20~5.12
  이모티콘: 0.0~1.0 → -0.27~3.68

입력 크기: 11663 (TF-IDF 30000 + 핸드크래프트 6, 정규화됨)


In [ ]:
class MLP(nn.Module):
    def __init__(self,i,h,o,d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [ ]:
sweep_config={
    'method':'bayes',
    'metric':{'name':'best_dev_accuracy','goal':'maximize'},
    'parameters':{
        'hidden_size':{'values':[256,512,1000]},
        'learning_rate':{'distribution':'log_uniform_values','min':1e-5,'max':1e-3},
        'dropout':{'values':[0.1,0.2,0.3,0.4]},
        'weight_decay':{'values':[0,1e-5,1e-4]},
        'batch_size':{'values':[128,256]},
        'num_epochs':{'values':[30,50,70]},
    }
}
sweep_id=wandb.sweep(sweep_config,project='nlp-hw1')
print(f'Sweep ID:{sweep_id}')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: 53rn3k4b
Sweep URL: https://wandb.ai/imeanseo_/nlp-hw1/sweeps/53rn3k4b
Sweep ID:53rn3k4b


In [ ]:
def train_sweep():
    run=wandb.init()
    cfg=run.config
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model=MLP(input_size,cfg.hidden_size,output_size,cfg.dropout).to(device)
    opt=optim.Adam(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    lfn=nn.CrossEntropyLoss()
    best_dev,best_state=0,None
    for epoch in range(cfg.num_epochs):
        model.train()
        total_loss=0
        for i in range(0,len(train_t),cfg.batch_size):
            bd=train_t[i:i+cfg.batch_size]
            bl=torch.tensor(train_labels[i:i+cfg.batch_size],device=device)
            loss=lfn(model(bd),bl)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss+=loss.item()
        model.eval()
        with torch.no_grad():
            da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
        if da>best_dev:
            best_dev,best_state=da,copy.deepcopy(model.state_dict())
        wandb.log({'epoch':epoch+1,
                   'train_loss':total_loss/max(1,len(train_t)//cfg.batch_size),
                   'dev_accuracy':da,'best_dev_accuracy':best_dev})
    model.load_state_dict(best_state)
    with torch.no_grad():
        test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
    wandb.log({'test_accuracy':test_acc})
    print(f'[Exp3-v9] Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')
    wandb.finish()

wandb.agent(sweep_id,train_sweep,count=15)

wandb: Agent Starting Run: erx9l418 with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.2
wandb: 	hidden_size: 512
wandb: 	learning_rate: 0.00040943471518030834
wandb: 	num_epochs: 50
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: aileen02-ko (imeanseo_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[Exp3-v9] Dev:0.6868|Test:68.13%


best_dev_accuracy,▁███████████████████████████████████████
dev_accuracy,▁███▇▇▆▆▆▆▆▅▇▆▆▅▆▆▆▆▆▆▇▆▆▅▆▆▆▆▆▆▆▆▆▆▆▆▆▅
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
test_accuracy,▁
train_loss,█▅▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.68684
dev_accuracy,0.65668
epoch,50
test_accuracy,0.68127
train_loss,0.68785


wandb: Agent Starting Run: 36vu38zo with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.2
wandb: 	hidden_size: 256
wandb: 	learning_rate: 0.00015322180875892405
wandb: 	num_epochs: 50
wandb: 	weight_decay: 1e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp3-v9] Dev:0.6916|Test:68.76%


...

[Exp3-v9] Dev:0.6897|Test:68.57%


best_dev_accuracy,▁▇██████████████████████████████████████
dev_accuracy,▆█▆▅▅▅▄▄▄▃▃▃▂▂▃▃▂▂▂▁▁▁▂▂▂▂▂▂▂▁▁▂▁▂▂▁▂▂▁▂
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
test_accuracy,▁
train_loss,█▅▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.68972
dev_accuracy,0.6634
epoch,70
test_accuracy,0.68569
train_loss,0.67677


wandb: Agent Starting Run: p80l05c6 with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.2
wandb: 	hidden_size: 512
wandb: 	learning_rate: 0.0004235898608971272
wandb: 	num_epochs: 70
wandb: 	weight_decay: 1e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp3-v9] Dev:0.6849|Test:68.17%


best_dev_accuracy,▁███████████████████████████████████████
dev_accuracy,▁█▆▆▄▅▅▅▅▅▅▄▄▄▄▄▅▅▅▄▃▄▄▅▄▄▃▄▄▄▄▄▅▅▃▄▄▄▄▄
epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇██
test_accuracy,▁
train_loss,█▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.68492
dev_accuracy,0.65379
epoch,70
test_accuracy,0.68165
train_loss,0.65366


In [ ]:
# Best config로 최종 학습
api=wandb.Api()
sweep=api.sweep(f'imeanseo_/nlp-hw1/{sweep_id}')
best_run=sorted(sweep.runs,key=lambda r:r.summary.get('best_dev_accuracy',0),reverse=True)[0]
cfg=best_run.config
print(f'Best config: hidden={cfg["hidden_size"]} | lr={cfg["learning_rate"]:.2e} | dropout={cfg["dropout"]} | epochs={cfg["num_epochs"]}')

torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
model=MLP(input_size,cfg['hidden_size'],output_size,cfg['dropout']).to(device)
opt=optim.Adam(model.parameters(),lr=cfg['learning_rate'],weight_decay=cfg['weight_decay'])
lfn=nn.CrossEntropyLoss()
best_dev,best_state=0,None
print('🏁 최종 학습...')
print('='*50)
for epoch in range(cfg['num_epochs']):
    model.train()
    for i in range(0,len(train_t),cfg['batch_size']):
        bd=train_t[i:i+cfg['batch_size']]
        bl=torch.tensor(train_labels[i:i+cfg['batch_size']],device=device)
        loss=lfn(model(bd),bl)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        da=(torch.argmax(model(dev_t),dim=1)==dev_labels_t).float().mean().item()
    if da>best_dev:
        best_dev,best_state=da,copy.deepcopy(model.state_dict())
        print(f'✨ Epoch {epoch+1}/{cfg["num_epochs"]}|Dev:{da:.4f}|NEW BEST!')
    elif (epoch+1)%10==0:
        print(f'Epoch {epoch+1}/{cfg["num_epochs"]}|Dev:{da:.4f}')
print('='*50)
model.load_state_dict(best_state)
torch.save(best_state,'best_model_exp3_v9_final.pt')
with torch.no_grad():
    test_acc=accuracy_score(test_labels_list,torch.argmax(model(test_t),dim=1).cpu().tolist())
print(f'\n✅ 저장: best_model_exp3_v9_final.pt')
print(f'📊 Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')
print(f'\n🎯 목표 70%: {"달성! 🎉🎉🎉" if test_acc>=0.70 else f"{test_acc*100:.2f}% (v8 대비 {(test_acc-0.6913)*100:+.2f}%p)"}')

Best config: hidden=256 | lr=1.53e-04 | dropout=0.2 | epochs=50
🏁 최종 학습...
✨ Epoch 1/50|Dev:0.5070|NEW BEST!
✨ Epoch 2/50|Dev:0.6334|NEW BEST!
✨ Epoch 3/50|Dev:0.6680|NEW BEST!
✨ Epoch 4/50|Dev:0.6857|NEW BEST!
✨ Epoch 5/50|Dev:0.6905|NEW BEST!
✨ Epoch 6/50|Dev:0.6916|NEW BEST!
Epoch 10/50|Dev:0.6747
Epoch 20/50|Dev:0.6626
Epoch 30/50|Dev:0.6594
Epoch 40/50|Dev:0.6567
Epoch 50/50|Dev:0.6578

✅ 저장: best_model_exp3_v9_final.pt
📊 Dev:0.6916|Test:68.76%

🎯 목표 70%: 68.76% (v8 대비 -0.37%p)


In [ ]:
from google.colab import files
files.download('best_model_exp3_v9_final.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>